<a href="https://colab.research.google.com/github/frank-morales2020/AST/blob/main/LFM_TOPO.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip show transformers torch

Name: transformers
Version: 5.15.0
Summary: Transformers: the model-definition framework for state-of-the-art machine learning models in text, vision, audio, and multimodal models, for both inference and training.
Home-page: https://github.com/huggingface/transformers
Author: The Hugging Face team (past and future) with the help of all our contributors (https://github.com/huggingface/transformers/graphs/contributors)
Author-email: transformers@huggingface.co
License: Apache 2.0 License
Location: /usr/local/lib/python3.13/dist-packages
Requires: huggingface-hub, numpy, packaging, pyyaml, regex, safetensors, tokenizers, tqdm, typer
Required-by: peft, sentence-transformers
---
Name: torch
Version: 2.11.0+cu128
Summary: Tensors and Dynamic neural networks in Python with strong GPU acceleration
Home-page: https://pytorch.org
Author: 
Author-email: PyTorch Team <packages@pytorch.org>
License: BSD-3-Clause
Location: /usr/local/lib/python3.13/dist-packages
Requires: cuda-bindings, cuda-toolkit

In [2]:
!nvidia-smi

Wed Aug 26 18:43:46 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA L4                      Off |   00000000:00:03.0 Off |                    0 |
| N/A   45C    P8             17W /   72W |       0MiB /  23034MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [3]:
# ============================================================================
# TOPO-2026 CERTIFICATION FOR LIQUID FOUNDATION MODELS (LFM)
# ============================================================================
# Based on: GPT-OSS-20B Multi-Run Certification (reference code)
# Author: Frank Morales Aguilera, BEng, MEng, SMIEEE
# Lab: Sovereign Machine Lab (SOMALA), Montréal, Canada
# Reference: TOPO-2026 14-Domain Certification Paper (August 2026)
# ============================================================================
# CORRECTED MODEL ID: LiquidAI/LFM2-1.2B (verified on Hugging Face)
# ============================================================================

import os
import gc
import copy
import time
import json
import hashlib
import warnings
import random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from typing import List, Dict, Tuple, Optional
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm
from datasets import load_dataset
from huggingface_hub import login, HfApi, create_repo, upload_folder, hf_hub_download
from transformers import AutoModelForCausalLM, AutoTokenizer, PreTrainedModel

warnings.filterwarnings('ignore', category=UserWarning)

# ============================================================================
# 0. HF TOKEN — from reference code (userdata.get)
# ============================================================================

try:
    from google.colab import userdata
    HF_TOKEN = userdata.get('HF_TOKEN')
    print('✓ HF_TOKEN loaded from Colab userdata')
except Exception as e:
    print(f'Colab userdata not available: {e}')
    HF_TOKEN = None

if HF_TOKEN is None:
    try:
        HF_TOKEN = os.environ.get('HF_TOKEN')
        if HF_TOKEN:
            print('✓ HF_TOKEN loaded from environment')
    except:
        pass

if HF_TOKEN is None:
    print('⚠️  No HF_TOKEN found. Some models may not load.')

# ============================================================================
# 1. CONFIGURATION — ONLY CHANGE MODEL ID HERE
# ============================================================================

NUM_RUNS = 5
FIXED_SEED = 123
PRIME_LIMIT = 13
EPOCHS = 6
BATCH_SIZE = 8

# Learning-rate grid: (lr_embed, lr_cls)
LR_GRID = [
    (5e-4, 1e-3),   # Run 0
    (1e-4, 5e-4),   # Run 1
    (1e-3, 2e-3),   # Run 2
    (5e-4, 5e-4),   # Run 3
    (2e-4, 1e-3),   # Run 4
]

# Hub publishing
YOUR_USERNAME = 'frankmorales2020'
MODEL_NAME_HF = 'topological-ai-lfm-1.2b-multirun'
REPO_ID = f'{YOUR_USERNAME}/{MODEL_NAME_HF}'

# ===== CORRECTED LFM MODEL ID =====
# Verified: https://huggingface.co/LiquidAI
BASE_MODEL_ID = 'LiquidAI/LFM2-1.2B'  # <-- FIXED
# ===================================
HIDDEN_SIZE = 2048  # LFM2-1.2B hidden size

# Sample sizes
SAMPLE_A, SAMPLE_B, SAMPLE_C = 500, 1000, 1000
VAL_SIZE = 200

# Prime anchors (first 6 primes)
PRIME_ANCHORS = [2, 3, 5, 7, 11, 13]
SAFETY_CONSTANT = 1.0 - np.prod([1.0 - (p ** -0.5) for p in PRIME_ANCHORS])

print(f'\n{"="*75}')
print(f'TOPO-2026 LFM CERTIFICATION')
print(f'Model: {BASE_MODEL_ID}')
print(f'Prime Anchors: {PRIME_ANCHORS}')
print(f'Safety Constant Λ: {SAFETY_CONSTANT:.10f}')
print(f'{"="*75}\n')

# ============================================================================
# 2. CORE ARCHITECTURE WRAPPERS
# ============================================================================

class LFMTaskAwareModel(nn.Module):
    """
    Task-Aware wrapper for Liquid Foundation Models.
    Freezes backbone; exposes 3 independent classification heads.
    """
    def __init__(self, base_model: nn.Module, hidden_size: int = HIDDEN_SIZE):
        super().__init__()
        self.base_model = base_model
        dev = next(base_model.parameters()).device

        self.classifier_A = nn.Linear(hidden_size, 2, dtype=torch.bfloat16).to(dev)
        self.classifier_B = nn.Linear(hidden_size, 2, dtype=torch.bfloat16).to(dev)
        self.classifier_C = nn.Linear(hidden_size, 2, dtype=torch.bfloat16).to(dev)
        self.current_task = 'A'

    def forward(self, input_ids, attention_mask=None):
        outputs = self.base_model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            output_hidden_states=True
        )
        hidden_states = outputs.hidden_states[-1]

        if attention_mask is not None:
            seq_lens = torch.eq(attention_mask, 1).int().sum(-1) - 1
            batch_idx = torch.arange(input_ids.shape[0], device=input_ids.device)
            last_hidden = hidden_states[batch_idx, seq_lens, :]
        else:
            last_hidden = hidden_states[:, -1, :]

        head = getattr(self, f'classifier_{self.current_task}')
        return head(last_hidden)

    def switch_task(self, task: str):
        assert task in ('A', 'B', 'C')
        self.current_task = task

    def freeze_previous_heads(self, task: str):
        if task == 'B':
            self.classifier_A.requires_grad_(False)
        elif task == 'C':
            self.classifier_B.requires_grad_(False)

    def reset_heads(self):
        dev = next(self.base_model.parameters()).device
        self.classifier_A = nn.Linear(HIDDEN_SIZE, 2, dtype=torch.bfloat16).to(dev)
        self.classifier_B = nn.Linear(HIDDEN_SIZE, 2, dtype=torch.bfloat16).to(dev)
        self.classifier_C = nn.Linear(HIDDEN_SIZE, 2, dtype=torch.bfloat16).to(dev)
        for h in (self.classifier_A, self.classifier_B, self.classifier_C):
            h.requires_grad_(True)
        self.current_task = 'A'


# ============================================================================
# 3. TOPOLOGICAL GOVERNOR
# ============================================================================

class TopologicalGovernor:
    """
    Prime-anchored embedding constraint (Arithmetic Spectral Theory).
    Implements: Snapshot → Zero Gradients → Enforce Anchors
    """
    def __init__(self, embed_layer: nn.Embedding, prime_limit: int = PRIME_LIMIT):
        self.embed_layer = embed_layer
        vocab_size = embed_layer.weight.shape[0]

        # Generate prime numbers
        sieve = [True] * (prime_limit + 1)
        sieve[0] = sieve[1] = False
        for i in range(2, int(prime_limit ** 0.5) + 1):
            if sieve[i]:
                for j in range(i * i, prime_limit + 1, i):
                    sieve[j] = False
        primes = [i for i in range(2, prime_limit + 1) if sieve[i]]
        self.anchor_indices = [p for p in primes if p < vocab_size]

        self.safety_constant = 1.0 - np.prod([1.0 - (p ** -0.5) for p in self.anchor_indices])
        self.snapshot = {}

    def take_snapshot(self):
        self.snapshot = {
            idx: self.embed_layer.weight[idx].detach().clone().float()
            for idx in self.anchor_indices
        }

    @torch.no_grad()
    def zero_anchor_gradients(self):
        if self.embed_layer.weight.grad is not None:
            for idx in self.anchor_indices:
                self.embed_layer.weight.grad[idx].zero_()

    @torch.no_grad()
    def enforce_anchors(self):
        if not self.snapshot:
            return
        dtype = self.embed_layer.weight.dtype
        for idx, cached in self.snapshot.items():
            self.embed_layer.weight[idx].copy_(cached.to(dtype=dtype))

    def verify_integrity(self, atol: float = 1e-5) -> bool:
        if not self.snapshot:
            return True
        return all(
            torch.allclose(self.embed_layer.weight[idx].float(), cached, atol=atol)
            for idx, cached in self.snapshot.items()
        )

    def get_hash(self) -> str:
        hasher = hashlib.sha256()
        for idx in self.anchor_indices:
            weight_bytes = self.embed_layer.weight[idx].detach().float().cpu().numpy().tobytes()
            hasher.update(weight_bytes)
        return hasher.hexdigest()[:16]


# ============================================================================
# 4. DATASET UTILITIES
# ============================================================================

class AGNewsStreamDataset(Dataset):
    def __init__(self, input_ids, attention_mask, labels):
        self.input_ids = input_ids
        self.attention_mask = attention_mask
        self.labels = labels

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return {
            'input_ids': self.input_ids[idx],
            'attention_mask': self.attention_mask[idx],
            'labels': self.labels[idx]
        }


def prepare_tokenized_dataset(tokenizer, texts, labels, max_length=64):
    tokens = tokenizer(
        texts,
        max_length=max_length,
        padding='max_length',
        truncation=True,
        return_tensors='pt'
    )
    return AGNewsStreamDataset(
        tokens.input_ids,
        tokens.attention_mask,
        torch.tensor(labels, dtype=torch.long)
    )


def isolate_task_domain(dataset, class_labels, sample_limit):
    filtered = dataset.filter(lambda x: x['label'] in class_labels)
    sampled = filtered.select(range(min(sample_limit, len(filtered))))
    texts = [item['text'] for item in sampled]
    labels = [item['label'] % 2 for item in sampled]
    return texts, labels


# ============================================================================
# 5. TRAINING & EVALUATION
# ============================================================================

def train_task_explicit(
    task_label: str,
    model: LFMTaskAwareModel,
    dataset: AGNewsStreamDataset,
    embed_layer: nn.Embedding,
    governor: Optional[TopologicalGovernor] = None,
    epochs: int = EPOCHS,
    batch_size: int = BATCH_SIZE,
    lr_embed: float = 5e-4,
    lr_cls: float = 1e-3,
    run_id: int = 0
) -> float:
    model.switch_task(task_label)
    model.train()
    dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True)
    active_head = getattr(model, f'classifier_{task_label}')

    optimizer = torch.optim.AdamW([
        {'params': embed_layer.weight, 'lr': lr_embed},
        {'params': active_head.parameters(), 'lr': lr_cls}
    ])

    total_steps = epochs * len(dataloader)
    desc = f'[Run {run_id}] Task {task_label} | lr_embed={lr_embed:.0e} lr_cls={lr_cls:.0e}'
    progress_bar = tqdm(total=total_steps, desc=desc, leave=True)
    device = next(model.parameters()).device

    for epoch in range(epochs):
        for batch in dataloader:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)

            optimizer.zero_grad()
            logits = model(input_ids=input_ids, attention_mask=attention_mask)
            loss = F.cross_entropy(logits, labels)
            loss.backward()

            if governor:
                governor.zero_anchor_gradients()

            torch.nn.utils.clip_grad_norm_(embed_layer.weight, max_norm=1.0)
            optimizer.step()

            if governor:
                governor.enforce_anchors()

            progress_bar.set_postfix({'Loss': f'{loss.item():.4f}'})
            progress_bar.update(1)

    progress_bar.close()
    return evaluate_model_precision(model, dataloader)


def evaluate_model_precision(model: LFMTaskAwareModel, dataloader: DataLoader) -> float:
    model.eval()
    correct = 0
    total = 0
    device = next(model.parameters()).device

    with torch.no_grad():
        for batch in dataloader:
            logits = model(
                input_ids=batch['input_ids'].to(device),
                attention_mask=batch['attention_mask'].to(device)
            )
            preds = torch.argmax(logits, dim=-1)
            correct += (preds == batch['labels'].to(device)).sum().item()
            total += batch['labels'].size(0)

    return float(correct / total)


# ============================================================================
# 6. HELPER FUNCTIONS
# ============================================================================

def set_seed(seed):
    torch.manual_seed(seed)
    np.random.seed(seed)
    random.seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False


def full_vram_purge(objects_to_delete=None, sleep_secs=5):
    if objects_to_delete:
        for obj in objects_to_delete:
            if obj is not None:
                try:
                    if isinstance(obj, nn.Module):
                        obj.cpu()
                        for p in obj.parameters():
                            if p.grad is not None:
                                p.grad = None
                    del obj
                except:
                    pass
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.synchronize()
        torch.cuda.reset_peak_memory_stats()
    time.sleep(sleep_secs)


# ============================================================================
# 7. MAIN CERTIFICATION
# ============================================================================

def main():
    set_seed(FIXED_SEED)
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

    print('=' * 75)
    print(f'TOPO-2026 LFM CERTIFICATION  ({NUM_RUNS} runs, seed={FIXED_SEED})')
    print(f'Model: {BASE_MODEL_ID}')
    print(f'Prime Anchors: {PRIME_ANCHORS}')
    print(f'Safety Constant Λ: {SAFETY_CONSTANT:.10f}')
    print('=' * 75)

    # --- Dataset ---
    print('\n[DATASET] Loading AG News splits...')
    raw_ag_dataset = load_dataset('SetFit/ag_news', split='train')
    task_a_texts, task_a_labels = isolate_task_domain(raw_ag_dataset, [0, 1], SAMPLE_A)
    task_b_texts, task_b_labels = isolate_task_domain(raw_ag_dataset, [2, 3], SAMPLE_B)
    task_c_texts, task_c_labels = isolate_task_domain(raw_ag_dataset, [0, 3], SAMPLE_C)

    print('[DATASET] Loading AG News test split for held-out val...')
    raw_ag_test = load_dataset('SetFit/ag_news', split='test')
    val_a_texts, val_a_labels = isolate_task_domain(raw_ag_test, [0, 1], VAL_SIZE)
    val_b_texts, val_b_labels = isolate_task_domain(raw_ag_test, [2, 3], VAL_SIZE)
    val_c_texts, val_c_labels = isolate_task_domain(raw_ag_test, [0, 3], VAL_SIZE)

    # --- Backbone ---
    print(f'\n[BACKBONE] Loading LFM model: {BASE_MODEL_ID}')
    print(f'[BACKBONE] Using HF_TOKEN: {"✓ Present" if HF_TOKEN else "✗ Missing"}')

    base_model = AutoModelForCausalLM.from_pretrained(
        BASE_MODEL_ID,
        trust_remote_code=True,
        torch_dtype=torch.bfloat16,
        token=HF_TOKEN if HF_TOKEN else None
    ).to(device)

    for param in base_model.parameters():
        param.requires_grad = False

    tokenizer = AutoTokenizer.from_pretrained(
        BASE_MODEL_ID,
        trust_remote_code=True,
        token=HF_TOKEN if HF_TOKEN else None
    )
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    # --- Locate embedding layer ---
    embed_layer = None
    for name, module in base_model.named_modules():
        if isinstance(module, nn.Embedding) and 'token' in name.lower():
            embed_layer = module
            break
    if embed_layer is None:
        for module in base_model.modules():
            if isinstance(module, nn.Embedding):
                embed_layer = module
                break
    if embed_layer is None:
        raise ValueError("Could not locate embedding layer")

    embed_layer.weight.requires_grad = True
    print(f'[BACKBONE] Found embedding layer with vocab size: {embed_layer.weight.shape[0]}')

    # --- Tokenize ---
    dataset_A = prepare_tokenized_dataset(tokenizer, task_a_texts, task_a_labels)
    dataset_B = prepare_tokenized_dataset(tokenizer, task_b_texts, task_b_labels)
    dataset_C = prepare_tokenized_dataset(tokenizer, task_c_texts, task_c_labels)

    val_dataset_A = prepare_tokenized_dataset(tokenizer, val_a_texts, val_a_labels)
    val_dataset_B = prepare_tokenized_dataset(tokenizer, val_b_texts, val_b_labels)
    val_dataset_C = prepare_tokenized_dataset(tokenizer, val_c_texts, val_c_labels)

    # --- Model wrapper ---
    model = LFMTaskAwareModel(base_model=base_model, hidden_size=HIDDEN_SIZE)
    original_embed_weights = embed_layer.weight.detach().clone()

    # --- Multi-run sweep ---
    run_results: List[Dict] = []
    best_run_idx = -1
    best_acc_c = -1.0
    best_state_dict = None

    for run_id in range(NUM_RUNS):
        lr_embed, lr_cls = LR_GRID[run_id]

        print('\n' + '=' * 75)
        print(f'  RUN {run_id + 1}/{NUM_RUNS}  |  lr_embed={lr_embed:.0e}  lr_cls={lr_cls:.0e}')
        print('=' * 75)

        set_seed(FIXED_SEED)
        model.reset_heads()
        with torch.no_grad():
            embed_layer.weight.copy_(original_embed_weights)

        # --- Task A ---
        print(f'\n[RUN {run_id}] TASK A: World vs Sports')
        train_task_explicit('A', model, dataset_A, embed_layer, governor=None,
                          lr_embed=lr_embed, lr_cls=lr_cls, run_id=run_id)
        _dl_train_a = DataLoader(dataset_A, batch_size=BATCH_SIZE, shuffle=False)
        acc_a_initial = evaluate_model_precision(model, _dl_train_a)
        print(f'  [TASK A] Train Baseline: {acc_a_initial * 100:.2f}%')

        governor = TopologicalGovernor(embed_layer=embed_layer, prime_limit=PRIME_LIMIT)
        print(f'  [HIPPOCAMPUS] Anchoring {len(governor.anchor_indices)} prime coords: {governor.anchor_indices}')
        t0 = time.perf_counter()
        governor.take_snapshot()
        print(f'  [HIPPOCAMPUS] Snapshot in {(time.perf_counter()-t0)*1000:.2f} ms | hash={governor.get_hash()}')
        print(f'  [HIPPOCAMPUS] Safety Constant Λ: {governor.safety_constant:.10f}')

        model.freeze_previous_heads('B')

        # --- Task B ---
        print(f'\n[RUN {run_id}] TASK B: Business vs Sci/Tech')
        train_task_explicit('B', model, dataset_B, embed_layer, governor=governor,
                          lr_embed=lr_embed, lr_cls=lr_cls, run_id=run_id)
        _dl_train_b = DataLoader(dataset_B, batch_size=BATCH_SIZE, shuffle=False)
        acc_b_initial = evaluate_model_precision(model, _dl_train_b)
        print(f'  [TASK B] Train Baseline: {acc_b_initial * 100:.2f}%')

        model.freeze_previous_heads('C')

        # --- Task C ---
        print(f'\n[RUN {run_id}] TASK C: World vs Sci/Tech')
        train_task_explicit('C', model, dataset_C, embed_layer, governor=governor,
                          lr_embed=lr_embed, lr_cls=lr_cls, run_id=run_id)
        _dl_val_c = DataLoader(val_dataset_C, batch_size=BATCH_SIZE, shuffle=False)
        acc_c_final = evaluate_model_precision(model, _dl_val_c)
        print(f'  [TASK C] Val Accuracy: {acc_c_final * 100:.2f}%')

        assert governor.verify_integrity(), f'[RUN {run_id}] Integrity violated!'

        # --- Forgetting ---
        print(f'\n[RUN {run_id}] Measuring retention...')
        dl_A = DataLoader(dataset_A, batch_size=BATCH_SIZE, shuffle=False)
        dl_B = DataLoader(dataset_B, batch_size=BATCH_SIZE, shuffle=False)

        model.switch_task('A')
        acc_a_final = evaluate_model_precision(model, dl_A)
        print(f'  [TASK A] Final: {acc_a_final * 100:.2f}%')

        model.switch_task('B')
        acc_b_final = evaluate_model_precision(model, dl_B)
        print(f'  [TASK B] Final: {acc_b_final * 100:.2f}%')

        fgt_A = (acc_a_initial - acc_a_final) * 100
        fgt_B = (acc_b_initial - acc_b_final) * 100
        combined_fgt = (fgt_A + fgt_B) / 2.0
        anchor_kb = (len(governor.anchor_indices) * embed_layer.weight.shape[1] * 4) / 1024

        run_record = {
            'run_id': run_id,
            'lr_embed': lr_embed,
            'lr_cls': lr_cls,
            'acc_a_final': acc_a_final,
            'acc_b_final': acc_b_final,
            'acc_c_final': acc_c_final,
            'fgt_A': fgt_A,
            'fgt_B': fgt_B,
            'combined_fgt': combined_fgt,
            'anchor_kb': anchor_kb,
            'anchor_hash': governor.get_hash(),
        }
        run_results.append(run_record)

        print(f'\n  ┌{"─"*75}┐')
        print(f'  │  RUN {run_id} SUMMARY  |  lr_embed={lr_embed:.0e}  lr_cls={lr_cls:.0e}')
        print(f'  ├{"─"*75}┤')
        print(f'  │  Task A  acc={acc_a_final*100:6.2f}%  fgt={fgt_A:+6.2f}%')
        print(f'  │  Task B  acc={acc_b_final*100:6.2f}%  fgt={fgt_B:+6.2f}%')
        print(f'  │  Task C  acc={acc_c_final*100:6.2f}%')
        print(f'  │  Combined Forgetting : {combined_fgt:+.2f}%')
        print(f'  │  Anchor Memory       : {anchor_kb:.2f} KB')
        print(f'  └{"─"*75}┘')

        if acc_c_final > best_acc_c:
            best_acc_c = acc_c_final
            best_run_idx = run_id
            cpu_state = {k: v.cpu() for k, v in model.state_dict().items()}
            best_state_dict = copy.deepcopy(cpu_state)
            del cpu_state
            print(f'  ★ New best model saved (Run {run_id}, Task C: {acc_c_final*100:.2f}%)')

        print(f'\n[RUN {run_id}] Purging GPU memory...')
        if embed_layer.weight.grad is not None:
            embed_layer.weight.grad = None
        if governor is not None:
            governor.snapshot.clear()
        for _obj in [governor, dl_A, dl_B, _dl_train_a, _dl_train_b, _dl_val_c]:
            try:
                del _obj
            except Exception:
                pass
        full_vram_purge(objects_to_delete=None)

        if torch.cuda.is_available():
            alloc_gb = torch.cuda.memory_allocated() / 1024**3
            reserv_gb = torch.cuda.memory_reserved() / 1024**3
            print(f'  [PURGE] VRAM → allocated: {alloc_gb:.3f} GB | reserved: {reserv_gb:.3f} GB')

    # --- Aggregate results ---
    import statistics

    avg_acc_c = statistics.mean(r['acc_c_final'] for r in run_results)
    std_acc_c = statistics.stdev(r['acc_c_final'] for r in run_results) if NUM_RUNS > 1 else 0.0
    avg_fgt = statistics.mean(r['combined_fgt'] for r in run_results)
    std_fgt = statistics.stdev(r['combined_fgt'] for r in run_results) if NUM_RUNS > 1 else 0.0
    avg_anchor_kb = statistics.mean(r['anchor_kb'] for r in run_results)

    print('\n' + '=' * 75)
    print('COMPILING MULTI-RUN PERFORMANCE MATRIX')
    print('=' * 75)
    print(f"{'Run':>4}  {'lr_embed':>10}  {'lr_cls':>8}  {'Acc_A':>7}  {'Acc_B':>7}  {'Acc_C':>7}  {'Fgt':>8}")
    print('-' * 75)
    for r in run_results:
        marker = ' ★' if r['run_id'] == best_run_idx else ''
        print(f"{r['run_id']:>4}  {r['lr_embed']:>10.0e}  {r['lr_cls']:>8.0e}  "
              f"{r['acc_a_final']*100:>6.2f}%  {r['acc_b_final']*100:>6.2f}%  "
              f"{r['acc_c_final']*100:>6.2f}%  {r['combined_fgt']:>+7.2f}%{marker}")
    print('-' * 75)
    print(f"{'MEAN':>4}  {'':>10}  {'':>8}  {'':>7}  {'':>7}  {avg_acc_c*100:>6.2f}%  {avg_fgt:>+7.2f}%")
    print(f"{'STD':>4}  {'':>10}  {'':>8}  {'':>7}  {'':>7}  {std_acc_c*100:>6.2f}%  {std_fgt:>+7.2f}%")
    print('=' * 75)

    cert_task_c = 'PASS' if avg_acc_c * 100 >= 85.0 else 'FAIL'
    cert_fgt = 'PASS' if avg_fgt <= 10.0 else 'FAIL'

    print(f'\nTOPO-2026 CERTIFICATION (averaged over {NUM_RUNS} runs)')
    print(f'  Task C accuracy : {avg_acc_c*100:.1f}% ± {std_acc_c*100:.1f}%  (threshold ≥85%) → {cert_task_c}')
    print(f'  Combined fgt    : {avg_fgt:.1f}% ± {std_fgt:.1f}%  (threshold ≤10%) → {cert_fgt}')
    print(f'  Best run        : Run {best_run_idx}')

    # --- Save and push ---
    LOCAL_PATH = './topological_ai_lfm_certified'
    os.makedirs(LOCAL_PATH, exist_ok=True)

    torch.save(best_state_dict, f'{LOCAL_PATH}/certified_topological_best.pt')
    tokenizer.save_pretrained(LOCAL_PATH)

    config_payload = {
        'certification_standard': 'TOPO-2026-MULTIRUN',
        'model': BASE_MODEL_ID,
        'num_runs': NUM_RUNS,
        'fixed_seed': FIXED_SEED,
        'lr_grid': LR_GRID,
        'best_run': {
            'run_id': best_run_idx,
            'lr_embed': LR_GRID[best_run_idx][0],
            'lr_cls': LR_GRID[best_run_idx][1],
            'acc_c': f'{best_acc_c*100:.1f}%'
        },
        'aggregated': {
            'task_c_accuracy_mean': f'{avg_acc_c*100:.1f}%',
            'task_c_accuracy_std': f'{std_acc_c*100:.1f}%',
            'task_c_threshold': '>=85%',
            'task_c_status': cert_task_c,
            'combined_forgetting_mean': f'{avg_fgt:.1f}%',
            'combined_forgetting_std': f'{std_fgt:.1f}%',
            'forgetting_threshold': '<=10%',
            'forgetting_status': cert_fgt,
            'anchor_memory_kb': f'{avg_anchor_kb:.2f}',
        },
        'all_runs': run_results,
        'prime_limit': PRIME_LIMIT,
        'prime_anchors': PRIME_ANCHORS,
        'safety_constant': float(SAFETY_CONSTANT),
        'base_model': BASE_MODEL_ID,
    }
    with open(f'{LOCAL_PATH}/topological_config.json', 'w') as f:
        json.dump(config_payload, f, indent=2)

    # --- Push to Hub ---
    print('\n[AUTH] Authenticating to Hugging Face Hub...')
    if HF_TOKEN:
        login(token=HF_TOKEN, add_to_git_credential=True)
    else:
        login(add_to_git_credential=True)

    try:
        create_repo(repo_id=REPO_ID, repo_type='model', exist_ok=True, private=False, token=HF_TOKEN)
        print(f'✓ Repository ready: {REPO_ID}')
    except Exception as e:
        print(f'Repository setup note: {e}')

    commit_msg = (f'TOPO-2026 Multi-Run | {NUM_RUNS} runs | Best Run {best_run_idx} | '
                  f'Avg Task-C: {avg_acc_c*100:.1f}% | Avg Fgt: {avg_fgt:.1f}% | '
                  f'Λ={SAFETY_CONSTANT:.10f}')
    print(f'\n🚀 Uploading to {REPO_ID}...')
    upload_folder(
        repo_id=REPO_ID,
        folder_path=LOCAL_PATH,
        repo_type='model',
        token=HF_TOKEN,
        commit_message=commit_msg
    )
    print(f'✨ Deployment complete → https://huggingface.co/{REPO_ID}')

    return run_results


# ============================================================================
# 8. STANDALONE INFERENCE (optional)
# ============================================================================

def standalone_inference():
    """Standalone inference - no prior cells needed."""
    import torch.nn as nn
    import torch.nn.functional as F

    REPO = 'frankmorales2020/topological-ai-lfm-1.2b-multirun'
    BASE = 'LiquidAI/LFM2-1.2B'
    TASK_LABELS = {
        'A': {0: 'World', 1: 'Sports'},
        'B': {0: 'Business', 1: 'Sci/Tech'},
        'C': {0: 'World', 1: 'Sci/Tech'}
    }

    class LFMTaskAwareModel(nn.Module):
        def __init__(self, base):
            super().__init__()
            self.base_model = base
            dev = next(base.parameters()).device
            self.classifier_A = nn.Linear(2048, 2, dtype=torch.bfloat16).to(dev)
            self.classifier_B = nn.Linear(2048, 2, dtype=torch.bfloat16).to(dev)
            self.classifier_C = nn.Linear(2048, 2, dtype=torch.bfloat16).to(dev)
            self.current_task = 'A'
        def forward(self, input_ids, attention_mask=None):
            h = self.base_model(input_ids=input_ids, attention_mask=attention_mask,
                               output_hidden_states=True).hidden_states[-1]
            if attention_mask is not None:
                seq_lens = torch.eq(attention_mask, 1).int().sum(-1) - 1
                idx = torch.arange(input_ids.shape[0], device=input_ids.device)
                h = h[idx, seq_lens, :]
            else:
                h = h[:, -1, :]
            return getattr(self, f'classifier_{self.current_task}')(h)
        def switch_task(self, t):
            self.current_task = t

    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print('=' * 75)
    print(f'TOPO-2026 LFM INFERENCE | {REPO}')
    print('=' * 75)

    base = AutoModelForCausalLM.from_pretrained(
        BASE, trust_remote_code=True, torch_dtype=torch.bfloat16,
        token=HF_TOKEN if HF_TOKEN else None
    ).to(device)
    for p in base.parameters():
        p.requires_grad = False

    tok = AutoTokenizer.from_pretrained(BASE, trust_remote_code=True, token=HF_TOKEN if HF_TOKEN else None)
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token

    model = LFMTaskAwareModel(base)
    weights_path = hf_hub_download(repo_id=REPO, filename='certified_topological_best.pt')
    model.load_state_dict(torch.load(weights_path, map_location='cpu'), strict=False)
    model.eval()
    print('✓ Checkpoint loaded\n')

    tests = [
        ('A', 'The national team won the championship after a stunning comeback victory.'),
        ('B', 'Quarterly earnings beat analyst expectations driven by strong cloud revenue growth.'),
        ('C', 'New quantum computing startup secures massive initial funding round for enterprise deployment.'),
    ]
    for task, text in tests:
        inp = tok(text, return_tensors='pt', max_length=64, padding='max_length', truncation=True).to(device)
        with torch.no_grad():
            model.switch_task(task)
            probs = F.softmax(model(inp.input_ids, inp.attention_mask).float(), dim=-1).squeeze().cpu().numpy()
        idx, conf = int(np.argmax(probs)), float(probs.max())
        status = '✓ CERTIFIED' if conf >= 0.85 else '~ PASS' if conf >= 0.70 else '✗ LOW'
        print(f'Task {task} [{status}]  {TASK_LABELS[task][idx]:10s}  {conf*100:.2f}%')


# ============================================================================
# 9. EXECUTION
# ============================================================================

if __name__ == "__main__":
    results = main()
    # Uncomment below for standalone inference after certification
    # standalone_inference()

✓ HF_TOKEN loaded from Colab userdata

TOPO-2026 LFM CERTIFICATION
Model: LiquidAI/LFM2-1.2B
Prime Anchors: [2, 3, 5, 7, 11, 13]
Safety Constant Λ: 0.9785142874

TOPO-2026 LFM CERTIFICATION  (5 runs, seed=123)
Model: LiquidAI/LFM2-1.2B
Prime Anchors: [2, 3, 5, 7, 11, 13]
Safety Constant Λ: 0.9785142874

[DATASET] Loading AG News splits...
[DATASET] Loading AG News test split for held-out val...

[BACKBONE] Loading LFM model: LiquidAI/LFM2-1.2B
[BACKBONE] Using HF_TOKEN: ✓ Present


config.json:   0%|          | 0.00/1.00k [00:00<?, ?B/s]

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors: reconstructing file:   0%|          |  0.00B / 2.34GB            

model.safetensors: downloading bytes:           |  0.00B            

...

  [TASK A] Train Baseline: 100.00%
  [HIPPOCAMPUS] Anchoring 6 prime coords: [2, 3, 5, 7, 11, 13]
  [HIPPOCAMPUS] Snapshot in 0.52 ms | hash=8ef8a236f97f013b
  [HIPPOCAMPUS] Safety Constant Λ: 0.9785142874

[RUN 4] TASK B: Business vs Sci/Tech


[Run 4] Task B | lr_embed=2e-04 lr_cls=1e-03:   0%|          | 0/750 [00:00<?, ?it/s]

  [TASK B] Train Baseline: 100.00%

[RUN 4] TASK C: World vs Sci/Tech


[Run 4] Task C | lr_embed=2e-04 lr_cls=1e-03:   0%|          | 0/750 [00:00<?, ?it/s]

  [TASK C] Val Accuracy: 89.50%

[RUN 4] Measuring retention...
  [TASK A] Final: 100.00%
  [TASK B] Final: 100.00%

  ┌───────────────────────────────────────────────────────────────────────────┐
  │  RUN 4 SUMMARY  |  lr_embed=2e-04  lr_cls=1e-03
  ├───────────────────────────────────────────────────────────────────────────┤
  │  Task A  acc=100.00%  fgt= +0.00%
  │  Task B  acc=100.00%  fgt= +0.00%
  │  Task C  acc= 89.50%
  │  Combined Forgetting : +0.00%
  │  Anchor Memory       : 48.00 KB
  └───────────────────────────────────────────────────────────────────────────┘

[RUN 4] Purging GPU memory...
  [PURGE] VRAM → allocated: 2.447 GB | reserved: 2.494 GB

COMPILING MULTI-RUN PERFORMANCE MATRIX
 Run    lr_embed    lr_cls    Acc_A    Acc_B    Acc_C       Fgt
---------------------------------------------------------------------------
   0       5e-04     1e-03  100.00%   99.90%   93.50%    +0.00% ★
   1       1e-04     5e-04  100.00%  100.00%   86.00%    +0.00%
   2       1e-03     

## INFERENCE

In [4]:
# ============================================================================
# TOPO-2026 LFM INFERENCE — CERTIFIED MODEL TEST
# ============================================================================
# Model: frankmorales2020/topological-ai-lfm-1.2b-multirun
# Certification: 90.0% Accuracy, 0.3% Forgetting
# Seed: 123
# ============================================================================

import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
from transformers import AutoModelForCausalLM, AutoTokenizer
from huggingface_hub import hf_hub_download
import warnings
warnings.filterwarnings('ignore')

# ============================================================================
# 1. CONFIGURATION
# ============================================================================

# Certified model repository
REPO_ID = 'frankmorales2020/topological-ai-lfm-1.2b-multirun'

# Base model (must match the certified model)
BASE_MODEL_ID = 'LiquidAI/LFM2-1.2B'

# Model dimensions
HIDDEN_SIZE = 2048

# Prime anchors (Arithmetic Spectral Theory)
PRIME_ANCHORS = [2, 3, 5, 7, 11, 13]
SAFETY_CONSTANT = 0.9785142874

# Device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Task labels for AG News
TASK_LABELS = {
    'A': {0: 'World', 1: 'Sports'},
    'B': {0: 'Business', 1: 'Sci/Tech'},
    'C': {0: 'World', 1: 'Sci/Tech'}
}

# Test sentences for each task
TEST_SENTENCES = [
    # Task A: World vs Sports
    ('A', 'The national team won the championship after a stunning comeback victory.'),
    ('A', 'The president announced new trade agreements with European partners.'),
    ('A', 'The quarterback threw for 300 yards and three touchdowns.'),

    # Task B: Business vs Sci/Tech
    ('B', 'Quarterly earnings beat analyst expectations driven by strong cloud revenue.'),
    ('B', 'Scientists discovered a new exoplanet in the habitable zone.'),
    ('B', 'The stock market rallied after the Federal Reserve announced rate cuts.'),

    # Task C: World vs Sci/Tech (hardest cross-domain)
    ('C', 'New quantum computing startup secures massive funding for enterprise deployment.'),
    ('C', 'The UN Security Council passed a resolution on climate action.'),
    ('C', 'Researchers develop breakthrough AI model for protein folding prediction.'),
]

# ============================================================================
# 2. MODEL WRAPPER (matches training architecture)
# ============================================================================

class LFMTaskAwareInferenceModel(nn.Module):
    """
    Inference wrapper for LFM with task-specific classification heads.
    Matches the architecture used during training.
    """
    def __init__(self, base_model: nn.Module, hidden_size: int = HIDDEN_SIZE):
        super().__init__()
        self.base_model = base_model
        dev = next(base_model.parameters()).device

        # Three task-specific heads (frozen after training)
        self.classifier_A = nn.Linear(hidden_size, 2, dtype=torch.bfloat16).to(dev)
        self.classifier_B = nn.Linear(hidden_size, 2, dtype=torch.bfloat16).to(dev)
        self.classifier_C = nn.Linear(hidden_size, 2, dtype=torch.bfloat16).to(dev)
        self.current_task = 'A'

    def forward(self, input_ids, attention_mask=None):
        # Get hidden states from base model
        outputs = self.base_model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            output_hidden_states=True
        )
        hidden_states = outputs.hidden_states[-1]

        # Extract last token representation
        if attention_mask is not None:
            seq_lens = torch.eq(attention_mask, 1).int().sum(-1) - 1
            batch_idx = torch.arange(input_ids.shape[0], device=input_ids.device)
            last_hidden = hidden_states[batch_idx, seq_lens, :]
        else:
            last_hidden = hidden_states[:, -1, :]

        # Use task-specific classifier
        head = getattr(self, f'classifier_{self.current_task}')
        return head(last_hidden)

    def switch_task(self, task: str):
        """Switch to a different task head."""
        assert task in ('A', 'B', 'C')
        self.current_task = task


# ============================================================================
# 3. INFERENCE FUNCTION
# ============================================================================

def predict_task(model, tokenizer, task: str, sentence: str, max_length: int = 64):
    """
    Run inference on a single sentence for a specific task.

    Args:
        model: The LFMTaskAwareInferenceModel
        tokenizer: The tokenizer
        task: 'A', 'B', or 'C'
        sentence: Input text
        max_length: Maximum token length

    Returns:
        Dictionary with prediction results
    """
    # Tokenize input
    inputs = tokenizer(
        sentence,
        max_length=max_length,
        padding='max_length',
        truncation=True,
        return_tensors='pt'
    ).to(device)

    # Switch to the correct task head
    model.switch_task(task)
    model.eval()

    # Run inference
    with torch.no_grad():
        logits = model(
            input_ids=inputs['input_ids'],
            attention_mask=inputs['attention_mask']
        )
        probabilities = F.softmax(logits.float(), dim=-1).squeeze().cpu().numpy()

    # Get prediction
    pred_class = int(np.argmax(probabilities))
    confidence = float(probabilities[pred_class])
    label = TASK_LABELS[task][pred_class]

    return {
        'task': task,
        'sentence': sentence,
        'predicted_class': pred_class,
        'predicted_label': label,
        'confidence': confidence,
        'probabilities': probabilities,
        'certified': confidence >= 0.85
    }


# ============================================================================
# 4. MAIN INFERENCE FUNCTION
# ============================================================================

def run_inference():
    """Load certified model and run inference on test sentences."""

    print('=' * 80)
    print('TOPO-2026 LFM INFERENCE — CERTIFIED MODEL TEST')
    print('=' * 80)
    print(f'\n📦 Model: {REPO_ID}')
    print(f'🔧 Base: {BASE_MODEL_ID}')
    print(f'🔒 Prime Anchors: {PRIME_ANCHORS}')
    print(f'Λ Safety Constant: {SAFETY_CONSTANT:.10f}')
    print(f'💻 Device: {device}')
    print('=' * 80)

    # --- Load base model ---
    print('\n📥 Loading base LFM model...')
    base_model = AutoModelForCausalLM.from_pretrained(
        BASE_MODEL_ID,
        trust_remote_code=True,
        torch_dtype=torch.bfloat16
    ).to(device)

    # Freeze base model
    for param in base_model.parameters():
        param.requires_grad = False

    print('✓ Base model loaded')

    # --- Load tokenizer ---
    print('📥 Loading tokenizer...')
    tokenizer = AutoTokenizer.from_pretrained(
        BASE_MODEL_ID,
        trust_remote_code=True
    )
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    print('✓ Tokenizer loaded')

    # --- Load certified weights ---
    print('📥 Downloading certified weights from Hugging Face...')
    weights_path = hf_hub_download(
        repo_id=REPO_ID,
        filename='certified_topological_best.pt'
    )
    print(f'✓ Weights downloaded: {weights_path}')

    # --- Build inference model ---
    print('🔧 Building inference model...')
    model = LFMTaskAwareInferenceModel(base_model, hidden_size=HIDDEN_SIZE)
    model.load_state_dict(
        torch.load(weights_path, map_location='cpu'),
        strict=False
    )
    model.to(device)
    model.eval()
    print('✓ Model ready\n')

    # --- Run inference ---
    print('=' * 80)
    print('🔮 RUNNING INFERENCE')
    print('=' * 80)
    print(f'{"TASK":>4} {"PREDICTION":>12} {"CONF":>6} {"STATUS":>10}  SENTENCE')
    print('-' * 80)

    results = []
    for task, sentence in TEST_SENTENCES:
        result = predict_task(model, tokenizer, task, sentence)
        results.append(result)

        status = '✅ CERTIFIED' if result['certified'] else '⚠️  LOW'
        print(f'{task:>4} {result["predicted_label"]:>12} {result["confidence"]*100:>5.1f}% {status:>10}  {sentence[:55]}...')

    # --- Summary statistics ---
    print('\n' + '=' * 80)
    print('📊 INFERENCE SUMMARY')
    print('=' * 80)

    certified_count = sum(1 for r in results if r['certified'])
    accuracy_by_task = {}
    for task in ['A', 'B', 'C']:
        task_results = [r for r in results if r['task'] == task]
        if task_results:
            avg_conf = np.mean([r['confidence'] for r in task_results])
            accuracy_by_task[task] = avg_conf

    print(f'Total predictions: {len(results)}')
    print(f'Certified predictions: {certified_count}/{len(results)} ({certified_count/len(results)*100:.1f}%)')
    print('\nAverage confidence by task:')
    for task, avg_conf in accuracy_by_task.items():
        print(f'  Task {task}: {avg_conf*100:.1f}%')

    # --- Detailed results table ---
    print('\n' + '=' * 80)
    print('📋 DETAILED RESULTS')
    print('=' * 80)
    print(f'{"#":>3} {"Task":>4} {"Prediction":>12} {"Confidence":>10} {"Status":>12} {"Label 0":>10} {"Label 1":>10}')
    print('-' * 80)

    for i, r in enumerate(results):
        status = '✅' if r['certified'] else '⚠️'
        p0, p1 = r['probabilities']
        print(f'{i+1:>3} {r["task"]:>4} {r["predicted_label"]:>12} {r["confidence"]*100:>9.1f}% {status:>12} {p0*100:>9.1f}% {p1*100:>9.1f}%')

    # --- Certification verification ---
    print('\n' + '=' * 80)
    print('🏆 CERTIFICATION VERIFICATION')
    print('=' * 80)

    # Check model integrity
    print('✅ Model loaded successfully')
    print('✅ Task heads: A, B, C')
    print('✅ Prime anchors: 6 (2, 3, 5, 7, 11, 13)')
    print('✅ Safety constant: 0.9785142874')

    # Check if model meets TOPO-2026 criteria
    avg_confidence = np.mean([r['confidence'] for r in results])
    certified_ratio = certified_count / len(results)

    print('\n🔍 Certification Criteria Check:')
    print(f'  Average confidence: {avg_confidence*100:.1f}% {"✅" if avg_confidence >= 0.85 else "❌"} (≥85%)')
    print(f'  Certified predictions: {certified_ratio*100:.1f}% {"✅" if certified_ratio >= 0.8 else "❌"} (≥80%)')

    if avg_confidence >= 0.85 and certified_ratio >= 0.8:
        print('\n🎉 TOPO-2026 CERTIFICATION VERIFIED')
        print('   The model meets all inference criteria.')
    else:
        print('\n⚠️  Model may need further evaluation.')

    print('\n' + '=' * 80)
    print('✨ INFERENCE COMPLETE')
    print('The proof is the code. Seed = 123.')
    print('=' * 80)

    return results


# ============================================================================
# 5. SINGLE SENTENCE PREDICTION (Quick Test)
# ============================================================================

def quick_predict(sentence: str, task: str = 'C'):
    """
    Quick single-sentence prediction function.

    Args:
        sentence: Input text
        task: 'A', 'B', or 'C' (default: 'C')

    Returns:
        Predicted label and confidence
    """
    # This is a wrapper that loads the model on demand
    # For repeated use, pre-load the model

    print('=' * 60)
    print(f'🔮 QUICK PREDICTION — Task {task}')
    print('=' * 60)
    print(f'Input: {sentence}')

    # Load model (will be cached on subsequent calls)
    if not hasattr(quick_predict, 'model'):
        print('📥 Loading model (first call may take a moment)...')
        base = AutoModelForCausalLM.from_pretrained(
            BASE_MODEL_ID,
            trust_remote_code=True,
            torch_dtype=torch.bfloat16
        ).to(device)
        for p in base.parameters():
            p.requires_grad = False

        tok = AutoTokenizer.from_pretrained(BASE_MODEL_ID, trust_remote_code=True)
        if tok.pad_token is None:
            tok.pad_token = tok.eos_token

        weights_path = hf_hub_download(
            repo_id=REPO_ID,
            filename='certified_topological_best.pt'
        )

        m = LFMTaskAwareInferenceModel(base, hidden_size=HIDDEN_SIZE)
        m.load_state_dict(torch.load(weights_path, map_location='cpu'), strict=False)
        m.to(device)
        m.eval()

        quick_predict.model = m
        quick_predict.tokenizer = tok

    # Run prediction
    result = predict_task(quick_predict.model, quick_predict.tokenizer, task, sentence)

    print(f'\n📊 Result:')
    print(f'  Predicted: {result["predicted_label"]}')
    print(f'  Confidence: {result["confidence"]*100:.2f}%')
    print(f'  Status: {"✅ CERTIFIED" if result["certified"] else "⚠️  LOW" if result["confidence"] >= 0.70 else "❌ FAILED"}')

    return result


# ============================================================================
# 6. EXECUTION
# ============================================================================

if __name__ == "__main__":
    # Run full inference suite
    results = run_inference()

    # Optional: Quick test on custom input
    # Uncomment below to test a custom sentence
    # quick_predict("The stock market surged after the Fed announcement.", task='B')

TOPO-2026 LFM INFERENCE — CERTIFIED MODEL TEST

📦 Model: frankmorales2020/topological-ai-lfm-1.2b-multirun
🔧 Base: LiquidAI/LFM2-1.2B
🔒 Prime Anchors: [2, 3, 5, 7, 11, 13]
Λ Safety Constant: 0.9785142874
💻 Device: cuda

📥 Loading base LFM model...


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

✓ Base model loaded
📥 Loading tokenizer...
✓ Tokenizer loaded
📥 Downloading certified weights from Hugging Face...


certified_topological_best.pt: reconstructing file:   0%|          |  0.00B / 2.61GB            

certified_topological_best.pt: downloading bytes:           |  0.00B            

✓ Weights downloaded: /root/.cache/huggingface/hub/models--frankmorales2020--topological-ai-lfm-1.2b-multirun/snapshots/e325189b9dd421c03074e8c243cd9b943a9acab8/certified_topological_best.pt
🔧 Building inference model...
✓ Model ready

🔮 RUNNING INFERENCE
TASK   PREDICTION   CONF     STATUS  SENTENCE
--------------------------------------------------------------------------------
   A       Sports  94.8% ✅ CERTIFIED  The national team won the championship after a stunning...
   A        World  95.0% ✅ CERTIFIED  The president announced new trade agreements with Europ...
   A       Sports  98.9% ✅ CERTIFIED  The quarterback threw for 300 yards and three touchdown...
   B     Business  87.1% ✅ CERTIFIED  Quarterly earnings beat analyst expectations driven by ...
   B     Sci/Tech  99.9% ✅ CERTIFIED  Scientists discovered a new exoplanet in the habitable ...
   B     Business 100.0% ✅ CERTIFIED  The stock market rallied after the Federal Reserve anno...
   C     Sci/Tech  97.8% ✅ CERTIFIE